# RU OCR: load probe on Kaggle (PP-OCRv6 small)

Short training runs on a 1 000-line subset to find settings that keep the GPU busy:
`num_workers`, `first_bs`, `max_w`, AMP. Each run logs GPU utilisation / memory (`nvidia-smi`),
CPU load (`psutil`) and PaddleOCR's `avg_reader_cost` (waiting for data) vs `avg_batch_cost` (whole step).
If reader cost is a large part of the step, the CPU-side augmentation is the bottleneck.

Settings: **Accelerator: GPU** (T4 x2 or P100), **Internet: on**, add the dataset made from
`ru_ocr_probe_subset.zip` (`PaddleOCR/data/kaggle_subset/`).

In [ ]:
import os, subprocess, sys
print("CPU cores:", os.cpu_count())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout.splitlines()[2])

## 1. Paddle GPU, PaddleOCR branch, pretrained weights
The wheel index must match the CUDA of the image (`cu126` for CUDA 12.x; try `cu118` if it does not import).

In [ ]:
PADDLE_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"
!pip install -q paddlepaddle-gpu==3.3.1 -i {PADDLE_INDEX}
import paddle
paddle.utils.run_check()

In [ ]:
%cd /kaggle/working
!git clone -q --depth 1 --branch feat/ru-ppocrv6-small-training https://github.com/ErshovVE/PaddleOCR.git
%cd /kaggle/working/PaddleOCR
!pip install -q -r requirements.txt
!mkdir -p pretrain && wget -q -O pretrain/PP-OCRv6_small_rec_pretrained.pdparams https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_rec_pretrained.pdparams
!ls -la pretrain

In [ ]:
import glob
DATA = glob.glob("/kaggle/input/**/ru_root", recursive=True)[0]
print(DATA, sum(1 for _ in open(f"{DATA}/labels/train_w_h.txt")), "train lines")

## 2. Probe runner
One run = `epochs` passes over the subset (a few hundred steps), no eval, then a summary row.
The first `SKIP` logged lines (warm-up, worker start) are left out of the averages.

In [ ]:
import re, threading, time, psutil, pandas as pd

CONFIG = "ru_ocr/configs/ru_PP-OCRv6_small_rec.yml"
SKIP = 3  # logged lines (print_batch_step=5 -> 15 steps) skipped as warm-up
STEP_RE = re.compile(r"avg_reader_cost: ([\d.]+) s, avg_batch_cost: ([\d.]+) s, avg_samples: ([\d.]+), ips: ([\d.]+)")
MEM_RE = re.compile(r"max_mem_reserved: (\d+) MB")


def sample_load(stop, rows):
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip().splitlines()
        gpu = [list(map(float, line.split(","))) for line in out]
        rows.append({"gpu_util": max(g[0] for g in gpu), "gpu_mem": max(g[1] for g in gpu),
                     "cpu": psutil.cpu_percent(interval=None)})
        time.sleep(2)


def probe(name, workers=4, bs=64, max_w=1600, amp=False, epochs=6, gpus="0", amp_dtype="float16"):
    # amp_dtype: float16 (T4 and older: loss scaling on) or bfloat16 (Ampere and newer: no scaling needed)
    two = "," in gpus
    opts = [
        f"Global.distributed={'true' if two else 'false'}", f"Global.epoch_num={epochs}",
        "Global.print_batch_step=5", "Global.eval_batch_step=[0,100000000]", "Global.save_epoch_step=100000",
        f"Global.save_model_dir=/kaggle/working/probe/{name}",
        "Global.pretrained_model=./pretrain/PP-OCRv6_small_rec_pretrained.pdparams",
        f"Train.dataset.data_dir={DATA}", f"Train.dataset.label_file_list=[{DATA}/labels/train_w_h.txt]",
        f"Eval.dataset.data_dir={DATA}", f"Eval.dataset.label_file_list=[{DATA}/labels/val_w_h.txt]",
        f"Train.loader.num_workers={workers}", f"Train.sampler.first_bs={bs}", f"Train.loader.batch_size_per_card={bs}",
        f"Train.sampler.max_w={max_w}", f"Global.use_amp={'true' if amp else 'false'}",
    ]
    if amp:  # PaddleOCR's AMP defaults to fp16 without loss scaling -- small gradients would vanish
        opts.append(f"Global.amp_dtype={amp_dtype}")
        if amp_dtype == "float16":
            opts += ["Global.scale_loss=1024", "Global.use_dynamic_loss_scaling=true"]
    head = [sys.executable, "-m", "paddle.distributed.launch", "--gpus", gpus] if two else [sys.executable]
    cmd = head + ["tools/train.py", "-c", CONFIG, "-o", *opts]
    load, stop = [], threading.Event()
    sampler = threading.Thread(target=sample_load, args=(stop, load))
    sampler.start()
    psutil.cpu_percent(interval=None)
    start = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True,
                          env={**os.environ, "CUDA_VISIBLE_DEVICES": gpus})
    stop.set()
    sampler.join()
    log = proc.stdout + proc.stderr
    for worker_log in glob.glob("log/workerlog.*"):  # paddle.distributed.launch writes the training log here
        log += open(worker_log).read()
    open(f"/kaggle/working/probe_{name}.log", "w").write(log)
    steps = [tuple(map(float, m)) for m in STEP_RE.findall(log)][SKIP:]
    mem = [int(m) for m in MEM_RE.findall(log)]
    busy = load[len(load) // 5:] or load
    row = {"name": name, "workers": workers, "bs": bs, "max_w": max_w, "amp": amp, "gpus": gpus,
           "ok": proc.returncode == 0 and bool(steps), "minutes": round((time.time() - start) / 60, 1)}
    if steps:
        reader = sum(s[0] for s in steps) / len(steps)
        batch = sum(s[1] for s in steps) / len(steps)
        row.update(reader_s=round(reader, 3), batch_s=round(batch, 3), reader_share=round(reader / batch, 2),
                   ips=round(sum(s[3] for s in steps) / len(steps), 1))
    if mem:
        row["max_mem_reserved_mb"] = max(mem)
    if busy:
        row.update(gpu_util=round(sum(r["gpu_util"] for r in busy) / len(busy)),
                   gpu_mem_peak=max(r["gpu_mem"] for r in load),
                   cpu=round(sum(r["cpu"] for r in busy) / len(busy)))
    if not row["ok"]:
        lines = log.strip().splitlines()
        row["error"] = lines[-1][:200] if lines else "no output"
    print(row)
    return row

## 3. Runs
`first_bs` is the batch of the lowest scale (h 32) and does not shrink with width: every batch is
padded to its longest line (up to `max_w`), and the subset is full of long lines -- the worst case
for memory. So first the largest batch that fits, then workers / AMP / width around it.
A failed run (out of memory, AMP error) stays in the table with its last log line.

In [ ]:
results = []
for bs in (8, 16, 24, 32, 48):
    row = probe(f"w4_bs{bs}", workers=4, bs=bs, epochs=3)
    results.append(row)
    if not row["ok"]:
        break
fits = [r for r in results if r["ok"]]
best_bs = max(r["bs"] for r in fits) if fits else None
print("largest first_bs that fits at max_w 1600:", best_bs)
pd.DataFrame(results)

In [ ]:
for workers in (2, 3):  # 4 is measured above
    results.append(probe(f"w{workers}_bs{best_bs}", workers=workers, bs=best_bs))
results.append(probe(f"w4_bs{best_bs}_amp", workers=4, bs=best_bs, amp=True))
results.append(probe(f"w4_bs{best_bs * 2}_amp", workers=4, bs=best_bs * 2, amp=True))  # AMP: ~half the activations
results.append(probe(f"w4_bs{max(8, best_bs // 2)}_mw3200", workers=4, bs=max(8, best_bs // 2), max_w=3200))
best_w = max((r for r in results if r["ok"]), key=lambda r: r["ips"])["workers"]
pd.DataFrame(results)

## 4. Two GPUs (T4 x2 only) and the summary file

In [ ]:
if subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ") > 1:
    results.append(probe(f"w{best_w}_bs{best_bs}_2gpu", workers=best_w, bs=best_bs, gpus="0,1"))
df = pd.DataFrame(results)
df.to_csv("/kaggle/working/probe_results.csv", index=False)
df

**How to read it.** `reader_share` near 0 and `gpu_util` above ~85 % — the GPU is the limit: raise `bs` while
memory allows (`max_mem_reserved_mb`). `reader_share` above ~0.3, or `gpu_util` well below 80 % with `cpu`
near 100 % — the CPU (augmentation) is the limit: more workers do not help past the core count; the fix is a
machine with more cores or cheaper augmentation. Send back `probe_results.csv` and the `probe_*.log` of the best
and of any failed run.